# CreditNLP Fine-Tuning Notebook

**Project:** Fine-tune Mistral-7B to predict startup loan default risk from application narratives.

**Baseline:** 60% accuracy with few-shot prompting (100% recall, 46.7% precision)

**Target:** 75%+ accuracy with balanced precision/recall

**Method:** LoRA (Low-Rank Adaptation) - parameter-efficient fine-tuning

---

## Prerequisites
1. Runtime → Change runtime type → T4 GPU
2. Upload `synthetic_applications.jsonl` when prompted
3. Have your HuggingFace token ready

## Step 1: Install Dependencies

This takes ~3-5 minutes. You'll see some warnings - ignore them.

In [ ]:
%%capture
!pip install -U transformers datasets accelerate peft bitsandbytes trl huggingface_hub
!pip install -U torch

print("Installation complete!")

In [ ]:
# Verify GPU
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError("No GPU detected! Go to Runtime → Change runtime type → T4 GPU")

## Step 2: HuggingFace Authentication

In [ ]:
from huggingface_hub import login

# This will prompt you to enter your token
login()

## Step 3: Upload Your Data

Run this cell and upload `synthetic_applications.jsonl` from `C:\Projects\CreditNLP\data\`

In [ ]:
from google.colab import files
import json

print("Please upload synthetic_applications.jsonl")
uploaded = files.upload()

# Load and verify
filename = list(uploaded.keys())[0]
print(f"\nLoaded: {filename}")

data = []
with open(filename, 'r') as f:
    for line in f:
        data.append(json.loads(line))

print(f"Total samples: {len(data)}")
print(f"Label 0 (No Default): {sum(1 for d in data if d['default_label'] == 0)}")
print(f"Label 1 (Default): {sum(1 for d in data if d['default_label'] == 1)}")

## Step 4: Prepare Training Data

Convert to instruction-tuning format and split train/test (same split as baseline).

In [ ]:
from datasets import Dataset
from sklearn.model_selection import train_test_split

# Same split as baseline evaluation (seed 42, stratified)
train_data, test_data = train_test_split(
    data,
    test_size=0.2,
    random_state=42,
    stratify=[d['default_label'] for d in data]
)

print(f"Train: {len(train_data)} samples")
print(f"Test: {len(test_data)} samples")
print(f"Train default rate: {sum(1 for d in train_data if d['default_label'] == 1) / len(train_data):.1%}")
print(f"Test default rate: {sum(1 for d in test_data if d['default_label'] == 1) / len(test_data):.1%}")

In [ ]:
# Format for instruction tuning
def format_for_training(sample):
    """Convert to chat format for Mistral instruction tuning."""
    label = "DEFAULT" if sample['default_label'] == 1 else "NO_DEFAULT"

    # Mistral chat format
    text = f"""<s>[INST] You are a credit risk analyst. Based on the following startup loan application, predict whether the startup will default on the loan. Respond with exactly one word: DEFAULT or NO_DEFAULT.

APPLICATION:
{sample['application_text']}

PREDICTION: [/INST] {label}</s>"""

    return {"text": text}

# Apply formatting
train_formatted = [format_for_training(s) for s in train_data]
test_formatted = [format_for_training(s) for s in test_data]

# Convert to HuggingFace Dataset
train_dataset = Dataset.from_list(train_formatted)
test_dataset = Dataset.from_list(test_formatted)

print(f"\nTraining samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"\n--- Sample Training Example ---")
print(train_dataset[0]['text'][:500] + "...")

## Step 5: Load Model with 4-bit Quantization

This loads Mistral-7B in 4-bit precision to fit in T4's 16GB VRAM.

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig
)
import torch

model_id = "mistralai/Mistral-7B-Instruct-v0.3"

# 4-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

print("Loading model (this takes 2-3 minutes)...")
model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

print(f"\nModel loaded!")
print(f"Model memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

## Step 6: Configure LoRA

LoRA (Low-Rank Adaptation) adds small trainable matrices to specific layers. This lets us fine-tune with ~0.1% of the parameters.

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Prepare model for training
model = prepare_model_for_kbit_training(model)

# LoRA configuration
lora_config = LoraConfig(
    r=16,                       # Rank of the update matrices
    lora_alpha=32,              # Scaling factor
    lora_dropout=0.05,          # Dropout for regularization
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[            # Which layers to apply LoRA to
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ]
)

# Apply LoRA
model = get_peft_model(model, lora_config)

# Print trainable parameters
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params = sum(p.numel() for p in model.parameters())
print(f"\nTrainable parameters: {trainable_params:,} ({100 * trainable_params / total_params:.2f}%)")
print(f"Total parameters: {total_params:,}")

## Step 7: Training

Fine-tune on 400 training samples. Takes ~20-40 minutes on T4.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

# Training arguments
training_args = TrainingArguments(
    output_dir="./creditnlp-mistral-lora",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    warmup_ratio=0.1,
    logging_steps=10,
    save_strategy="epoch",
    fp16=True,
    optim="paged_adamw_8bit",
    report_to="none",  # Disable wandb
)

# Initialize trainer
trainer = SFTTrainer(
    model=model,
    train_dataset=train_dataset,
    args=training_args,
    tokenizer=tokenizer,
    max_seq_length=2048,
)

print("Starting training...")
print(f"Epochs: {training_args.num_train_epochs}")
print(f"Batch size: {training_args.per_device_train_batch_size}")
print(f"Gradient accumulation: {training_args.gradient_accumulation_steps}")
print(f"Effective batch size: {training_args.per_device_train_batch_size * training_args.gradient_accumulation_steps}")
print(f"Learning rate: {training_args.learning_rate}")
print("\n" + "="*50)

In [ ]:
# Run training
trainer.train()

print("\n" + "="*50)
print("Training complete!")

## Step 8: Save the Fine-Tuned Model

In [ ]:
# Save LoRA adapter weights
adapter_path = "./creditnlp-lora-adapter"
trainer.model.save_pretrained(adapter_path)
tokenizer.save_pretrained(adapter_path)

print(f"Adapter saved to: {adapter_path}")

# List saved files
import os
print("\nSaved files:")
for f in os.listdir(adapter_path):
    size = os.path.getsize(os.path.join(adapter_path, f)) / 1e6
    print(f"  {f}: {size:.1f} MB")

## Step 9: Evaluate on Test Set

Run predictions on the same 100 test samples used for baseline.

In [ ]:
from tqdm import tqdm

def predict(model, tokenizer, application_text):
    """Generate prediction for a single application."""
    prompt = f"""<s>[INST] You are a credit risk analyst. Based on the following startup loan application, predict whether the startup will default on the loan. Respond with exactly one word: DEFAULT or NO_DEFAULT.

APPLICATION:
{application_text}

PREDICTION: [/INST]"""

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=5,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    response = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract prediction from response
    response_part = response.split("PREDICTION:")[-1].strip()

    if "NO_DEFAULT" in response_part.upper():
        return 0
    elif "DEFAULT" in response_part.upper():
        return 1
    else:
        return -1  # Parsing error

print("Running evaluation on test set...")
results = []

for sample in tqdm(test_data, desc="Evaluating"):
    predicted = predict(model, tokenizer, sample['application_text'])
    actual = sample['default_label']
    results.append({
        'application_id': sample['application_id'],
        'actual': actual,
        'predicted': predicted,
        'correct': predicted == actual,
        'risk_score': sample['risk_score']
    })

print(f"\nEvaluation complete!")

In [ ]:
# Calculate metrics
import json

# Filter out parsing errors
valid_results = [r for r in results if r['predicted'] != -1]
parse_errors = len(results) - len(valid_results)

if parse_errors > 0:
    print(f"Warning: {parse_errors} parsing errors")

# Calculate confusion matrix
tp = sum(1 for r in valid_results if r['actual'] == 1 and r['predicted'] == 1)
tn = sum(1 for r in valid_results if r['actual'] == 0 and r['predicted'] == 0)
fp = sum(1 for r in valid_results if r['actual'] == 0 and r['predicted'] == 1)
fn = sum(1 for r in valid_results if r['actual'] == 1 and r['predicted'] == 0)

accuracy = (tp + tn) / len(valid_results)
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
recall = tp / (tp + fn) if (tp + fn) > 0 else 0
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0

print("="*50)
print("FINE-TUNED MODEL RESULTS")
print("="*50)
print(f"\nOverall Accuracy: {accuracy:.1%}")
print(f"\nDEFAULT Class Metrics:")
print(f"  Precision: {precision:.1%}")
print(f"  Recall: {recall:.1%}")
print(f"  F1 Score: {f1:.1%}")
print(f"\nConfusion Matrix:")
print(f"                 Predicted")
print(f"                 NO_DEF  DEFAULT")
print(f"  Actual NO_DEF    {tn:3d}      {fp:3d}")
print(f"  Actual DEFAULT   {fn:3d}      {tp:3d}")

# Compare to baseline
print("\n" + "="*50)
print("COMPARISON TO BASELINE")
print("="*50)
print(f"\n{'Metric':<20} {'Baseline':>12} {'Fine-Tuned':>12} {'Change':>12}")
print("-"*56)
print(f"{'Accuracy':<20} {'60.0%':>12} {f'{accuracy:.1%}':>12} {f'{(accuracy-0.60)*100:+.1f}pp':>12}")
print(f"{'Precision':<20} {'46.7%':>12} {f'{precision:.1%}':>12} {f'{(precision-0.467)*100:+.1f}pp':>12}")
print(f"{'Recall':<20} {'100.0%':>12} {f'{recall:.1%}':>12} {f'{(recall-1.0)*100:+.1f}pp':>12}")
print(f"{'F1 Score':<20} {'63.6%':>12} {f'{f1:.1%}':>12} {f'{(f1-0.636)*100:+.1f}pp':>12}")

In [ ]:
# Error analysis by risk score
print("\n" + "="*50)
print("ERROR ANALYSIS BY RISK SCORE")
print("="*50)

# False positives (predicted DEFAULT, actually safe)
fp_results = [r for r in valid_results if r['actual'] == 0 and r['predicted'] == 1]

# False negatives (predicted NO_DEFAULT, actually defaulted)
fn_results = [r for r in valid_results if r['actual'] == 1 and r['predicted'] == 0]

def count_by_risk_range(results_list):
    ranges = {
        "-100 to -50": 0,
        "-50 to 0": 0,
        "0 to 50": 0,
        "50 to 100": 0
    }
    for r in results_list:
        score = r['risk_score']
        if score <= -50:
            ranges["-100 to -50"] += 1
        elif score <= 0:
            ranges["-50 to 0"] += 1
        elif score <= 50:
            ranges["0 to 50"] += 1
        else:
            ranges["50 to 100"] += 1
    return ranges

print(f"\nFalse Positives: {len(fp_results)} (flagged safe startups as risky)")
if fp_results:
    fp_ranges = count_by_risk_range(fp_results)
    for range_name, count in fp_ranges.items():
        if count > 0:
            print(f"  Risk {range_name}: {count}")

print(f"\nFalse Negatives: {len(fn_results)} (missed actual defaults)")
if fn_results:
    fn_ranges = count_by_risk_range(fn_results)
    for range_name, count in fn_ranges.items():
        if count > 0:
            print(f"  Risk {range_name}: {count}")

## Step 10: Save Results and Download

In [ ]:
# Save detailed results
from datetime import datetime

evaluation_results = {
    "timestamp": datetime.now().isoformat(),
    "model": "mistralai/Mistral-7B-Instruct-v0.3 + LoRA",
    "test_set_size": len(valid_results),
    "metrics": {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1_score": f1,
        "confusion_matrix": {
            "true_positives": tp,
            "true_negatives": tn,
            "false_positives": fp,
            "false_negatives": fn
        }
    },
    "baseline_comparison": {
        "baseline_accuracy": 0.60,
        "accuracy_improvement": accuracy - 0.60,
        "baseline_precision": 0.467,
        "precision_improvement": precision - 0.467,
        "baseline_recall": 1.0,
        "recall_change": recall - 1.0,
        "baseline_f1": 0.636,
        "f1_improvement": f1 - 0.636
    },
    "detailed_results": results
}

with open("finetuned_evaluation.json", "w") as f:
    json.dump(evaluation_results, f, indent=2)

print("Results saved to finetuned_evaluation.json")

In [ ]:
# Create zip of adapter and results for download
import shutil

# Zip the adapter
shutil.make_archive("creditnlp-lora-adapter", "zip", adapter_path)
print("Created: creditnlp-lora-adapter.zip")

# Download files
print("\nDownloading files...")
files.download("finetuned_evaluation.json")
files.download("creditnlp-lora-adapter.zip")

print("\nDone! Save these files to C:\\Projects\\CreditNLP\\")

## Summary

You've now:
1. ✅ Fine-tuned Mistral-7B with LoRA on 400 startup applications
2. ✅ Evaluated on 100 test samples
3. ✅ Compared results to baseline
4. ✅ Downloaded the adapter weights and results

**Next Steps:**
1. Copy `finetuned_evaluation.json` to `C:\Projects\CreditNLP\results\`
2. Extract `creditnlp-lora-adapter.zip` to `C:\Projects\CreditNLP\models\`
3. Update PROJECT_LOG.md with Phase 3 results
4. Prepare presentation slides